In [19]:
import pandas as pd
from pathlib import Path
import sys

# Explicit mapping from the names in your test file to the actual .csv filenames
RENAME_DICT = {
    'InDelphi mESC': 'XCRISP_inDelphi_mESC',
    'InDelphi mESC NHEJdeficient': 'XCRISP_inDelphi_mESC_NHEJdeficient',
    'InDelphi U2OS': 'XCRISP_inDelphi_U2OS',
    'FORECasT 2A TREX2': 'FORECasT_K562_2A_TREX2',
    'FORECasT TREX2': 'FORECasT_K562_TREX2',
    'FORECasT eCAS9': 'FORECasT_K562_eCAS9',
    # Adding Aldit variants just in case they have spaces too
    'Aldit K562': 'ALDIT_K562',
    'Aldit Jurkat': 'ALDIT_Jurkat',
    'Aldit HAP1': 'ALDIT_HAP1',
    'Aldit Jurkat DNTTKO': 'ALDIT_Jurkat_DNTTKO',
    'Aldit K562 DNTTOE': 'ALDIT_K562_DNTTOE'
}

def load_required_datasets(test_file_path, data_root="../data"):
    root = Path(data_root)
    test_path = Path(test_file_path)
    
    # 1. Load test file and find unique datasets
    test_df = pd.read_csv(test_path)
    raw_names = test_df['dataset_name'].dropna().unique()
    
    dataset_cache = {}
    missing = []

    print(f"--- 📂 Loading Datasets for {test_path.name} ---")

    for name in raw_names:
        # Check dictionary first, then fallback to replacing spaces with underscores
        target_stem = RENAME_DICT.get(name, name.replace(" ", "_"))
        
        # Try finding the file (case-insensitive glob to handle ALDIT vs Aldit)
        file_matches = list(root.rglob(f"{target_stem}.csv"))
        
        # If no match, try case-insensitive lookup
        if not file_matches:
            file_matches = [f for f in root.rglob("*.csv") if f.stem.lower() == target_stem.lower()]

        if file_matches:
            file_path = file_matches[0]
            try:
                # Cache only unique sequences to save memory
                df = pd.read_csv(file_path, usecols=['sequence'])
                dataset_cache[name] = set(df['sequence'].dropna().unique())
                sys.stdout.write(f"\r[Matched] {name} -> {file_path.name}{' ' * 10}")
                sys.stdout.flush()
            except Exception as e:
                print(f"\n⚠️ Error loading {file_path.name}: {e}")
        else:
            missing.append(name)

    print(f"\n\n✅ Cache ready: {len(dataset_cache)} datasets.")
    if missing:
        print(f"❌ Still Missing: {missing}")

    return dataset_cache

def map_test_to_original_targeted(test_file_path, dataset_cache):
    test_df = pd.read_csv(test_file_path)
    # We use drop_duplicates so we don't process the same seq/dataset pair twice
    tasks = test_df[['sequence', 'dataset_name']].drop_duplicates()
    
    mapping_results = []
    
    print(f"--- 🧬 Mapping {len(tasks)} unique pairs ---")
    
    for i, (idx, row) in enumerate(tasks.iterrows(), 1):
        ts = row['sequence']
        ds_name = row['dataset_name']
        
        if ds_name not in dataset_cache:
            continue
            
        sys.stdout.write(f"\r[Matching] {i}/{len(tasks)}: {ds_name[:30]}...")
        sys.stdout.flush()

        source_seqs = dataset_cache[ds_name]
        ts_clean = str(ts).replace("P", "").upper()
        
        # Suffix matching logic
        matches = [
            orig for orig in source_seqs 
            if str(orig).replace("P", "").upper().endswith(ts_clean)
        ]

        if len(matches) == 1:
            mapping_results.append({
                'dataset': ds_name,
                'sequence': ts,
                'padded/processed': matches[0]
            })
        elif len(matches) > 1:
            print(f"\n\n❌ Ambiguity Error in {ds_name} for sequence {ts}")
            raise ValueError(f"Matches: {matches}")

    print("\n✅ Mapping Complete.")
    return pd.DataFrame(mapping_results)

# --- RUN ---
test_file = "tests/test_CROTON_APINDEL.csv"
cache = load_required_datasets(test_file, data_root="../data")
df_mapping = map_test_to_original_targeted(test_file, cache)

--- 📂 Loading Datasets for test_CROTON_APINDEL.csv ---
[Matched] SPROUT T -> SPROUT_T.csv          2OS.csv          _NHEJdeficient.csv          

✅ Cache ready: 18 datasets.
--- 🧬 Mapping 52280 unique pairs ---
[Matching] 52279/52280: SPROUT T...OS...EJdeficient...
✅ Mapping Complete.


In [22]:
print(df_mapping.head())
# save the mapping for later use
df_mapping.to_csv("mapping_CROTON_APINDEL.csv", index=False)

         dataset                                           sequence  \
0  FORECasT K562  PPPPPPPPTAAAGGGATTGTTCCGTGCTGACATAAGGTAGACCTCT...   
1  FORECasT K562  TGTAAAGCATCGTCATGTAACCTTTTTTTCAGTCGAGTGTGACACT...   
2  FORECasT K562  PPPPPPPPPPPPPPPPPPPPCACAGCCCGAACATAACGCTCGTTGC...   
3  FORECasT K562  PPPPPPPPAAAATCCTTGCTCTATCATCAGCGAGTCCACCAGCACT...   
4  FORECasT K562  PPPPPPPPAAAGATTTCCGTCGCATGGGCGCTCATAACAGCTGTTA...   

                                    padded/processed  
0  TAAAGGGATTGTTCCGTGCTGACATAAGGTAGACCTCTTCGGCGGC...  
1  TAATGCTGTAAAGCATCGTCATGTAACCTTTTTTTCAGTCGAGTGT...  
2  CACAGCCCGAACATAACGCTCGTTGCTAACTGGAAGGCGTATCAGG...  
3  AAAATCCTTGCTCTATCATCAGCGAGTCCACCAGCACTGCTTAGGT...  
4  AAAGATTTCCGTCGCATGGGCGCTCATAACAGCTGTTAAATACGGT...  


In [21]:
print(len(df_mapping))

52279


In [23]:
test_file = "tests/test_ex1_seed1.csv"
cache = load_required_datasets(test_file, data_root="../data")
df_mapping = map_test_to_original_targeted(test_file, cache)
print(df_mapping.head())
# save the mapping for later use
df_mapping.to_csv("mapping_ex1_seed1.csv", index=False)
print(len(df_mapping))

--- 📂 Loading Datasets for test_ex1_seed1.csv ---
[Matched] InDelphi U2OS -> XCRISP_inDelphi_U2OS.csv          _NHEJdeficient.csv          

✅ Cache ready: 18 datasets.
--- 🧬 Mapping 50840 unique pairs ---
[Matching] 50840/50840: InDelphi U2OS...EJdeficient...
✅ Mapping Complete.
         dataset                                           sequence  \
0  FORECasT K562  AGGGATGCCAGGAGATCGACCAGCTACGACAACGAGTAGTCCTACA...   
1  FORECasT K562  PPPPPPPPPPPPPPPPPPPPGACCAGTCATCTACCATACGAATGAA...   
2  FORECasT K562  PPPPPPPPCAATCGTACGTATTTCTGCGCCTTGAACACACACTGAC...   
3  FORECasT K562  PPPPPPPPAAACAACCCCCTAGAGGCGTTTTGTGACCTCCCGCCCG...   
4  FORECasT K562  PPPPPPPPTAACTTTGGAAACAATTAGCTAGTGTCGGTGGTAAATT...   

                                    padded/processed  
0  GAAGTAAGGGATGCCAGGAGATCGACCAGCTACGACAACGAGTAGT...  
1  GACCAGTCATCTACCATACGAATGAATTAGAGGTCCACTCACATGA...  
2  CAATCGTACGTATTTCTGCGCCTTGAACACACACTGACCATTGGGA...  
3  AAACAACCCCCTAGAGGCGTTTTGTGACCTCCCGCCCGCGGTAGGG...  
4  TAACTTTGGAAACA